In [ ]:
import polars as pl

pl.Config.set_tbl_rows(99)
pl.Config.set_tbl_cols(99)
pl.Config.set_tbl_width_chars(999)

In [ ]:
def clean(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(pl.col("name").str.to_titlecase())


df_batch_measurements = pl.scan_parquet("../data/batch_measurements.parquet")
df_dim_vet = pl.scan_parquet("../data/dim_vet.parquet")
df_visitors = pl.read_parquet("../data/visitors.parquet")

print(df_dim_vet.head(2).collect())
print(df_visitors.head(2))

How many times has vet practice Digital Wildlife Care diagnosed a polar bear as sick?


In [ ]:
practice = "Digital Wildlife Care"

len(
    df_batch_measurements.pipe(clean)
    .join(df_dim_vet, on="vet", how="left", suffix="_vet")
    .filter(pl.col("practice") == practice)
    .filter(pl.col("vet_health_check") == "SICK")
    .collect()
)

Which vet(s) has/have never seen a weight above the 99.9th percentile? (Hint: the keyword to search for is `quantile`.)


In [ ]:
df_vet_seen_bigboi = (
    df_batch_measurements.pipe(clean)
    .join(df_dim_vet, on="vet", how="left", suffix="_vet")
    .filter(pl.col("weight") > pl.col("weight").quantile(0.999))
    # somehow the 99.9th percentile is different than the one pyspark gives: pl.lit(876.13875)
    # .filter(pl.col("weight") > pl.lit(876.13875))
    .select("vet")
    .unique()
)

df_dim_vet.join(df_vet_seen_bigboi, on="vet", how="anti").collect()

Which vet was the least consistent in name capitalization?


In [ ]:
consistent_name = pl.col("name") == pl.col("name").str.to_titlecase()

(
    df_batch_measurements.join(df_dim_vet, on="vet", how="left", suffix="_vet")
    .group_by("name_vet")
    .agg(nr_consistent=consistent_name.sum(), nr_total=pl.len())
    .with_columns((pl.col("nr_consistent") / pl.col("nr_total") * 100).alias("%"))
    .sort("%")
).collect()

Which day had the lowest bear-to-visitor ratio? A bear can be seen if the vet determines the bear is healthy. (Hint: while joins require an exact match, this question is most easily solved using a non-exact match.)


In [ ]:
# Ensure 1 visitor record per day
assert len(df_visitors) == len(
    df_visitors.group_by(pl.col("timestamp").dt.date()).agg()
)

df_healthy_bears = (
    df_batch_measurements.pipe(clean)
    .with_columns(day=pl.col("timestamp").dt.date())
    .filter(pl.col("vet_health_check") == "HEALTHY")
    .group_by("day")
    .agg(healthy_bears=pl.len())
)

(
    df_visitors.lazy()
    .with_columns(day=pl.col("timestamp").dt.date())
    .join(df_healthy_bears, on="day", how="left")
    .sort("day")
    .with_columns(healthy_bears=pl.col("healthy_bears").forward_fill())
    .filter(pl.col("healthy_bears").is_not_null())
    .with_columns(bear_to_visitor_ratio=pl.col("healthy_bears") / pl.col("visitors"))
    .sort("bear_to_visitor_ratio")
).collect()